# Phase 10: Advanced Transformer Intent Model (DistilRoBERTa)
### Google Colab GPU Training & Golden Evaluation Pipeline

This notebook fine-tunes a pretrained **DistilRoBERTa** (`distilroberta-base`) model on the isolated AppleSupport intent training dataset and benchmarks performance against the human-verified **Golden Evaluation Set**.

**Key Constraints:**
- **Hardware Requirement:** Execution **MUST** run on a CUDA GPU (e.g. Tesla T4, V100, A100). CPU training is prohibited.
- **Strict Training Isolation:** Asserts $S_{\text{train}} \cap S_{\text{golden}} = \emptyset$.
- **Zero Data Leakage:** Only initial customer message text is used. No agent replies or future conversation turns.
- **Reproducibility:** Deterministic training with fixed seed (`42`).

In [ ]:
# Cell 1: Pre-Flight GPU Hardware Validation
import sys
import subprocess

print("=== Checking GPU Availability via nvidia-smi ===")
try:
    nvidia_smi = subprocess.check_output(["nvidia-smi"]).decode("utf-8")
    print(nvidia_smi)
except Exception as e:
    print(f"nvidia-smi failed: {e}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "CRITICAL ERROR: CUDA GPU is not available in this runtime!\n"
        "Please switch to a GPU runtime in Colab: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"\nSUCCESS: Connected to GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
print(f"PyTorch Version: {torch.__version__}, CUDA Version: {torch.version.cuda}")

In [ ]:
# Cell 2: Install Required Deep Learning Dependencies
!pip install -q --upgrade transformers datasets accelerate evaluate scikit-learn pandas scipy

In [ ]:
# Cell 3: Environment Setup, Repository Synchronization & Dataset Resolution
import os
import sys
import gzip
import base64
import shutil
import importlib
from pathlib import Path

REPO_NAME = "ai-customer-agent"
CURRENT_DIR = Path.cwd()

# 1. Ensure repository code directory structure is active
if not (CURRENT_DIR / "src").exists():
    if not Path(REPO_NAME).exists():
        print(f"Cloning {REPO_NAME} from GitHub...")
        !git clone https://github.com/mohanraj9342/ai-customer-agent.git
    os.chdir(REPO_NAME)

print(f"Active working directory: {os.getcwd()}")
Path("src/classification").mkdir(parents=True, exist_ok=True)
Path("models/distilroberta_intent_classifier").mkdir(parents=True, exist_ok=True)
TARGET_DATA_DIR = Path("data/processed/apple_support")
TARGET_DATA_DIR.mkdir(parents=True, exist_ok=True)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# 2. Synchronize / update Python helper modules
train_py = Path("src/classification/train_distilroberta.py")
eval_py = Path("src/classification/eval_transformer.py")

print("Deploying latest train_distilroberta.py and eval_transformer.py...")
train_py.write_bytes(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/61c/XLbSHL/n08xxy1nARcIW17vZcOEV6WVJa8Sra2S5L1cFBUOAoY0YhBgAFA2d+M/U5VXyWvlSdIf8wmAtFy3W3cWMOjp6emZ7vlN9wyn0+mkbbJnWZm2bbEssrQr6upZ16RFleRF2xVlU9/LpkvjzW6yeOx/k1dSbsRNk1btsm7WshFnRSVnN9uqqFbistjIEt5F8IpauKp/PL26SUMBtOJ4synl9XazqZtOnFedrDpx4kkXTyb/Infi9TYF/p2U7XwyE9ddU2SdeH35TpxWwCeTa6g5F8dtC9K3ooOi93G2zdO4aJP0IS3K9L6UQSjuJZBLQV1G6e7lqqjaeCLERZ2lpTgBluZj0YqWWip3YtPU74v7opO52EAPa/iH5IM6q22RUxeBj5HtRjM5b+uSKOfil7Qs8rSTrbhOqBHxf//zv/C8qstcVmIBr/+tJVyiCjtSIXI9rzZbYCo/deL0U1Zu2+Kh6HZz8bYC2aChrgBBsm3b1ah/EBa6uwbpoTVUTSuCv3ZQ+a+hSIH5tpU5ciVWOcjzbSVl3iaNfCjkx2/FT3Lb4GBl4qSuliX0p52Ln2voJeqERAc9GN3UlTg6ElldZY3sQLnpp7qq1zt4aFayI63QoM7+LIvV+078mJZplUHP5oJLgNlJUwPBadU19WYnSngRaZ43EkYURkE+SBCb5q0o1vdUXyLf46Yrlino+1I2LYgsoXwurmUD+ih+hapaj9DER2qrjWB6fJAVfG0iFHpZrCKRVrkd97XsUhinFBt4zWNzLUHxD2m5VUN5Uq9hQIB/BqLKfPaxbsocK4JKoAWYHbOCp3MjcXJDWQt6lJYEW0zX98VqW29bkRfpqqpR5aCuKdjpZNnUa5Eky223bWSSQLfJSNKqqjsSop1MdFmz2qRNK/X7f7R1pZ/LegUzfKVf61Y/tbuW28AZ2RVrqVvQ76An+PfXupJMt0m792Vxr8ku4ZU/dLsNWQuXH1e7SLyCiRGJCxiQCFT3n1sclkjcbMHajdTVdg0jnbai2uiiDSgFCuB/m5x5tx9KmTZVrNSmGwnAYIVIs2zbpNkuaTMwmYjKfN+WsPbVJxhrMBwoXafA7ROXbhqZFS3TggcokyVxS1r2StEk7EmCZpC0spQZtqAlYicKM6JL2k1ZdGoAwd3Gvkjx/bYoc2XziTRzClh2fvdOjq9Pk1fnZ2fnJ+8ubv6S3JyfXl2z0GWd7mHB38FciuUu0TM6KbQTst0ZCsYTNkHe4EOUKGDAifpQpWsJkw5nFHxf6KkVA8kFlQUJ0SRJOJm8Oj07BqmTm6vj8zfJyfUvUGGKRvUMnFOGZp0/S9H7a0X7b7pN04MMpgY7zzhrH6aG/+u3F69O/4YGlBJBcz7ft+9uLt/dgPqvkC8NevvMWyM1B61F2djap78cXyT/fP32zVcK5TfgjCw4wm0JrhRNe2q1+/PbV6cXyZvjn0+pIbf27D5tpZXo+vT0FdC8fGHrHv9rcnH65vXNT1B+9OIH8+HH45uTn5Lr839Dpt/ZChenx1dvzt+8Tq6Ob+iTnH1ve3z59uSnayw1RX8+vvr53SVSn7+FD8/jI/vp9Pz1T6Dd05Pjv9Cn50eTySSXS5pvsFiDia1BNTDyLazasz+x77hFv3IL604Ea153x37mFh4jXIzu7uY0+cF9XknwmpWASVNXBa7rxPJFkZPbLfIX9C50E4RFuvcSlzGzetG4wnRDb4xsXSsAmVsYL5kHffsIwpAtVDe4EL/hh7koqJUiEvgK3IQED4goQgYug/Azt6ZlhPrFnOt8Tf2GNaDFiAxDpWflIVabrYePQNNWx+DKrUp/oQqgpRQWHHHy7tUx4S/AA6Y+q9HBgdp6AcQhl6u0wNX8alvhynLaNNifJfMCPtvKcIr3wDGLwmItF/3tmh0LSqpTHhlBIBXKT5ncAL6kcm4WlhgotXUaFM2TLDDfqKHL3Q3yQyFgAQb1A7YqS0AVMBA4c2AlanC5l9VD0dQVgq5YTHs8YAWBVmTVwpIudvWWwFizrbh7zOgEPPU9qbZhaWLLBUAz+m6QnBUKykNh9sLdr+jfydX5zfnJ8cXcG1rkbvj94QCu5ukG6jhLy1YOe96DzA1ggqJhaIpW7GwMxBpQLKBgAzNhjU1ZngrQWL0qlY4GjTwawbsaZUXmAHwzmWQ1KAgszumn+yUIXWIyQ48WfYHzMXju0WfpJr0vABrs9tayJLpuB2APXKFc1w0Y6z3UbECSPBivD10EJXcFuKHnYezWFc9EcPT8xcunT78LI/FCdRutv4ANFDoZo5IpcrWDO53DXmar0BV9d1UCX93XIRUqwhLhm0OTMYp2+g2ky+lvA3XcPr/7HI8UH919njr8etoCZr0SjxY0mDzgtgGW1LkakUSXJAnTflZ4C/FNjLoKpjgXlSUJ8oqFzOfiCeyx4P+vfxTcWsSG5Iz6kzZ0hNX/uZoZiGtG6HZMV3eh6+o1qXLwgGthUyATRB+AbtpAOUqEqYB0EoTzc7QO8V+E5VkyBYfy5RwgePwK6p41ZswAjiRkwjBIgBI7XtSf88dWohYKsh8XdADitOu3x9NvQRxe22HZ28SwO2ia1FmULgCrWmu3GDECJ0txAWGRL2wFyw79EGyk/f0ub8YItjOzZ9DT2K4tqDJUF/QMNRX4Sgw9V2yIY/kJ3Fo7dMJnRSnf1N0Z2jF74uX0ZtgFgZAWWS6RUKTdXPxmuX+eKhP2ZiaqA7mMaIRWDpyl08gRUjmoJQzrR+gdDEgjYWcBXQtcIqL6RhzFOroBG6j3Ev4hbGVULLL3MvswPxDdOLw5CViQyM7CcNhHJYHj4nXrD8YWYWtfixrey3QjPhYwckoIhPlT058XsY6AAPYAhF2XD7DerAFoFzPYE8kVOs5MRUDA8X4EE/dnjtFfBks79pF7cKv+TI3+FUac3ok/AFB3eUzv4qze7IJhV62vOqOZi2shzN0neT9cg5L9o2ARTItiU9eleA9AByrA1rZucsCy1v+UUusb7FO/EY9wQOQWG+19ByDNQ9Ee7o1cBDsG6kc0R4+35mFEewA7iirQbcQf5A4Bt6dBW53IQOMO7zGWIJLh2GeBMbMeBy6KwZmUVRpMp2Gctt1uIwPwUkY3L2MdIRIqWJWr6JUKQhFUpigXF2CIxoQveC1tvWZVZ2LcEqrFFroet7yVzeUn1X3YyqZrcLVY2xs99VXtatRXv99atgX6WrShNjAVIpFjLxfwhXz/dy+4DvYjSwrCwYD8V9JWcTxfRsuC2zmELQFWjMRRaOiUBLf4AdVuewP4xQr/FBmO+Aal8/yQ0p/kdmcXWYWYofuenBx4FHQldjnASYNOhoM7apbQ1wifgAJRXS8IZE1YD4M1LaYqfpULs6xGzkpR5fU6ofjtAldW+6ll4XaLwdzQ9jlQyyuGACz7HDXA/lnpNsISt4tcjOpRM4jIQ/MKtKHSl8IefV1EI34g0sOgAMpI2uFLGMWBFiau5OGWL1SywSKuBe4U5gvI0OypYKNAXIGDf7yfxwqW0kZimFJu6ux92wdEHCVhivu0AwCKM6BPZQMwKtqHgUdcKXGzb7HXaFxGyZl+SmCwVqgJn7eN+nwFbKOYrgns2uCABWJnOsze28zBdEK4vG1xqf5pSwHDsxRwGSEeQGO0OpMNzj7qXAD6RQO/CHaoLA/xUjsTgsD9Xcx4VMNZ7wkwKjBgw2uIC3z4S+HNfTHWoDfb3CXxkmE3RtFTgpUskoUpo75j3F7YX1qvPA7phyaz8F+t7zDdW5gnx7HATHB8jerUIJiicwYkgf6sPIwlcKJAvaA9/ne87WpKJ53VjU4P+Gm/yKO9MQkbU6zmT68AZtdxs9pyXMHpGXjafsd0YYD/uAu3aewAvtYkClBbt6B37eo7DJonf4zKgW26VMGNwK2pggVLUz0hFxHQv3PH9DCvckv2tzdg5/hmI4wf8SGuGs7429JNmmM3F1PrRnob167ZVjxQCz844DufhX2M+iGXbzC3CMaBc8vzDGouMZwElMtr1UKXKxVSpsisTbeqI5HQi6HaG0N9Xpr31IaPo3WHY6/G0Gyv2LjtqqlNIo5xh+HKnsAIIJZTrwQ4/SGOeDTA/lCbruS2Lm5Jv1zTghiyr2E3zjlXzB1ple31smY8n1VIw8Cl0JvokY8fQ7s8qjl/2L4HhmCmh9OEKau2a94/tAuvVUuhnebCeE+7h1HUC+Ni/X3MHwGrU+LcX5PcjPUgP30BLxavmyy2YhCovw4ARrNOEkzUJwl4nHIZiadpswKAQMuq5+kBQFaA63VI6obeAJu8qSsMOOIfqP30w0fDgNwAls89Q/xGvNqBInGTXu5wm+Y4pn+CGionRek16keaNZR4L0vfgTeSItetx1159aJqNzLreu6B+g921qRrXLkUUdwWqyrFhLZWUayVEsZELGGf22tmibE6JfYUtxnccbK/ab8LROA3Px+E3JjB7bAybjn4Y7ypN4HTbugxkSVKNdq2K5wv9mOlsrX64gwaHJNqv66cL5zC6I/THo2OEg/lZ0/1GPXx8lsu48EExMD4B4+23W7AoEIzTwIyGzv/fc7EtQ+bxoxr4lmmjq0i8FTWqVwgWWeBh2/aeW+hjdQCm/AuAr7fY8RlwSmQMQuFv77a2KeRfSAH2hPzItQfW9UGkJJgwdOnXMWnAk9fEJEiVwyptMcQpsqIslTKhzwNzgbFcFDMcjvFw+mAukyWmU2nVFVMTlT5UHShAbe7GEoSd3XAjasMTBiOtkA4nRvS9BgTC2ZHEfeP1BXzMZ/YriOhQtyGPOzbUrunT1a7sPGu0Wl4ZAp0Bfgp0pQhqtufLdQCMbS4T09DddglwF0HrpG5nUJ2CtIu0EF73PvIzilT25Dgi4ruwMQEbBboSumnol04cZg0y4DOP1/Dyy4ebkI2lnaNq0ayPIIKy6MxUmAPOC9dSQSUQAs4AoNLsO1+oIM3i+f9AJDMH8lPkx9iqUbkNz9ZqPs2VfvpAApCH8dOdc8MjS7oEzpSG1qnzCH/rFHH38f2iKDZsjDyUIu2uNaLpTjn5ROhWfCuKjBFBaD2F85UUaQRQBXviEMd2PAyFjbQ4X+P1x+gLMBdJTTPWF5Q4iKpPzhglAL7m7oAImTCNsAtPBNT5+t04kYSlbM1LoBfA8/SdWCRKUxscbIHYgCC2I8n3O3fOLLgc2PglaEbWdd36X4e1CptStGeoKcFZ1inXogGyL13hw4WM52s5b2BjQJBLfsyXoVMek8N8VS8cGqhq+MWOBIFtPzg0PAIAPMsRTsYO5zjdhHjIfcYtWSInnaJrPJhepi9VwLY0aEGMmtODvGqkaCgJilaIO7gacivBWtPODFaFusCU85Hnli0dUzaTm6wl98/d+vC3h4Hz4ugTvlQIPBEqSpYvaY616uMc7k5+qNQZ7J0dm+KhYSpcArSedEvH7swM+2Wq+MMw+7phpQJz9p0Kb14FAZ55WoneA0wr9B6JT/iwVU3IuV8xbBOYzKSU6+2Fn5UPJ8S5ZzSfOEgHEPLkTa/yLNP73IeUwKOtum+6QfNgUe16VN+qTUAaOvtRgT8l6PwjpJVMU2tnnZ5QvKXBYY8AkAcAQXInW1/KJ49c2w0BCNlK7RD5DZ9sGce4Z0TrnXP2Tmj5Qr/GMZMeEfetQvc/j0dbcpsoDk7iRzVKtbV9tTNGjuvbQkPzxtfLOqlc4zK3W/qwwZFntDuBbzyh7l4oPTNhwgeMPqi5Y+LTq7xkCD0+YPq5mebR0DXoJgMlghA0bYV050rTgTr/Y4rcW6305Pfa5+reejNwqEVSbtS+us4NKyJR1jcDrtHXTjTwgEqPIzjzFCHjFyAR6UCUCMndhREBbJeiUs7su+CCn6Be8pm76Z639bT192eDX3nxXPHNsmPZe/vzPt83Q3DVwrmTgT40A8nPX3q8xsLT153aYNZ5F7+xblGonIxNj7Jc4BPFuvkJfCjvyPHEZxDKjUmCPEgnXhd1vcARclPYGYxotAYPMUvlyOHndwm4xXVJR8T+V/MRMb9kTHMH2JxjasDX0TBFdjE1c09kHZEM+kDCu1cBFkbBjYe1tUqlq+BbegOSkzLCm+9EQkaol68n+mcsGaPWPXkH1RPjEZ/VjdOOIqqXkZsHyw7ew8OL0PfgujFG20XWOHp78TGUrXT6J/AsztiAq2jsdWpLkMSL4ZKX3XEFb4Og69TM5Qqq2yODfqpZVvBhtT3VcH0s+u29uLbL2Prx8J2JxEyF2NJjanO2I8CTqMEPDMIK956g/5V3bOJq/pjoK/axNsuCwFO1rgUwi7S7ShmNZ3jxcBCZz7HmqLYhN6MHrAutwFrkFiVYMAegw1dv0271noDo+PuCnm+mbtUdHMhAlgyxcN5Wc0Zpm23nP0wDfE89tK6T6SN8+16E+jqkVhiHC6Hfi9e+GcPeDF0L3XpSuqggYKhtOPSGCOB6WWvfgQ2e8EhP9+qnbKD5yP19J/3zy5+RU7+0BF83Og38j2sm8WDt10AJIVHxx0X568C6SpFpEVE6kKbvcyGd9vsJYfRdG+bFZtdjNAGrxnqC2T1soM+TQaJc/PsHssyhc6hqkHZwVNVHMnDXnvnsDGeBZU5k+qAqyAMVehQe129WojltiydE4EiOPr+BzpEx3xpdsrcXaGDYQrdyNzVJSjbtVSdP/U3sntzpl/Il6q4IcOldjHddFP3pA/FgZOCAntK8tupKST5AhVC5eBHh+fe+BZc+8Gt5H/xa1o7V8HcOlk1ae7udfshaiPDwjxFvdYX/ms42RvIVrHdbLMNwphuDupjHXjXle/j0HQcDWli2BBPyUFX/PgnVfZpnQHGamCJVFed3lyIW23lt8Ud38hBCOm2cDexqUHbXpPugvG6fGwOFzf9OQzvbJacL5j+mS6YHh3NKC/pmi/M3u+FOb2qTnqqmYwAHj+7tuZ0UD0+8mwqHtndSsVPsx4w4ROKrapBeunXGOrVrcS3hhMTx1YXrcjI0FdbMUKNqDhcbeK4btDaUkeOODpfTREpQJ/80EQmmB2JhA/cHLobar3CeCtfDnw7dmzixZvIPjeRGxH/vYXaHz13M+RZuVTXaHUurX+99stNquS9N7SRPX6XdSr0vF892Rpb793gDb6mPWNR13gRe3YhH2ATcCkbQlp4CMLJfv8g8mIJvQTUsxNdoffqdIU72dg6Y9v1z+ZcLFakiIQzg0cv9YaOG8VKvXXUNdkMcb0VDl3Sgurc+RwofqNO+SqW1r3CFtglWojnfq4L9Ax7ya3sV0DSqbm0ntBJeW3DPoeBpm5JRm9L41/bwfiFkWm4c5xWNSH0qfJ8+PsDeBoOnsqdeChSvsqf03EW3NqtYezplE5a7trCO/puket4qg/MCUyrU5Ewrb6hu1Sa37NWhHG7XQe9vOLvrxclLEVi6GmExkl28fUx3cNnLm/xMuzrSFnMsfmdghM8BgKTXv9aAcx61q8IvvPXHpgmB6bxiCoPrDx2yunhnPMhuN5xVFye1eqL9pdEiOooWkjCYKiwQZwXePaWq4GGLwDqPkq80Zvry1UGPugDCrijJ8qRETc1EGECk08eGqZj+q4SbFs8lfK72+d39mTCJkHpF4xwbjVLx87rjQNqiBpxDXqbILydfTe/u53PZ0d+BYtKbr2x/m2q7ZhTXUwGOCWMcI5p0zIziBdbapSJXoafPYYG3jhiGgIrk2O1egVnpk410Mqdm28mwz5ILWZiWH6E5yXBk5FLtOWh+JM44nS8FcaOwnDuxekG9rq5f5xyzHz18KIF5yNmSRuIuZ1RvKEIRygHc23uTMpRpzAYMFs0NPRenYRVbKry655qqEklHv4oCd2kNNNshP79FvyesnJwPejWW1cJXHAX7nPX/uZ/ECBTwS4/MCYC59eJ3EuYborqb4rNuKkuvGWBIihLx4sew0uq6qNymSq4Ze+9eWF03AAk9Asz+iA6QkOvpgbXbkW8AIf40Mbq95+B4IGGguEoq7ytgZ09F6C+hnsrMkodrdXsr0UnKYY18DDFsIrFznuEtAQHq4+Kar8erDsQ2DsA4lX83Mvuq59vcHIqBnJ7qRcf/A7H0wRyfZzdV6+qnK1HQheuaIxVSsTJjmwDBOPmoYyvRJhqPSZUG7pRk3Pn68r4A0aUuOJfodC/aRS/wR+12KSZwmhUiEkSQ6DTeZf0JchlmzXFhuIszrUUhITaeEbvqOisCDcQpzmlBYlzMJ3NKBA6w5+IifCsVgrQdzG4i3SQAxv3Hhb2ZtJBHrxjmuGRlCEPe1npIA/Uwkz9nswIF/PTNQeZkLOdkbMdsrB3oQ4LwoH7SND5H4qWDoSh+1IHuVBcf0ZB/kOc7J2qg9x0AmBGCQHFkOx5yNK7d3VYW+mnmb5HcUBGGwc+yI1yDIf44P0t76cBFCPXxpTZrSnX5x8f17/rdJ+2RXZCJycD8gIL/eX8zdnbSPAyuJg+CdI2w0UybMXtEyalGy3tnXgS8NMcnsCO23QFL8rSVFrel4rxNsLx6b9Xi7/1P23TzFD4vzfoewH3Nwv/zo2v6R8w9Hn9HqLpuDRdJ+GkJPsiG+8ezWuYm6YjlydHboPRgWxTNrgN5tPZwqgX2cUjb0xj36ORaxtMM5ZtZIPn7/1knU3P8fexdJ2XnWOyPQk7J6TO0ozE1em+G30d3A37Ro+/pAExPwH4NXcEe9rkyZM87lo8oTnlolXk9DEJLDMOi94BETOHFt3wJt3BS4EH7tV8Scc23pb62SrdMVxkbHfd46o8QxwlhD5dzAdWH3t61eYnDYevTkS60vSSkWMnDmQ+1mNzyMDIYZR0iW5FtNs1bLb4cNFax8xV7dsBoL/zveVsNuv5NDe66U1kThp4qYUrFbwRwMZzdcvpsdonzM1md337rd48fHs3j18uP/eq/IxofXZ25FXREH68ij71YmphFQdEH2roUuP+udOQ2QwcqnlFkH/ui8j7AFttMinw+hb/uCHFqpIEV84kUWFPXkYn/w8KwPzkYVcAAA==")))
eval_py.write_bytes(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/51YW2/bOBZ+168gtCggY2RlOtiHnQBaIG3sIos0CWx3XoKAoCXK4UQStSSVNtvNf5/DmyT6kg3WKBqJPJfvXHmoOI4jKYqzoiZSsooVRDHentFnUmMlSCsrLhoqsu4lyt/1ixbA2hspiLQl+sybjgh4f6bokpFdy6ViBVq0O9ZSBNLREh7mm76lJdqMGtFXXtJaZlF0zUkpUaWplKG6ZCCiXvFPi9WGoEbTGVWKP9GW/Qd4K8EbVDL5lCJq4VCJAJB6pNFj35B2/kwFWAvCvvC6pC2aoF5TlRp5hYEOnCUTtFD1CyI7wlqp0N0jkRT9jrbwpwZgMkq+kj+5YOoFfXJrKdos51eXS/QLuoZXItD6j6/TRb5jxhUrugMlElTPsiiGeEQGPsZVr3pBMUas6bhQgKnlymCUUeTXxA4wSurf/5S89c8134GTd1ZcR9RjzbZe1h282g310gGRX79oX1JwcKEGDR24gkgE/7rSQYOEycKEybY9q0u8M87EdHAmllShAQ45QXFSKmQga3Fp4i34lgpFvLQkQvC7XCwvvl1v8Jfb68vFDf68/iMN1m+/be6+bfDl1cqu+2yY5jYGEA4WYLF0O6pwTba0xg3ptH9kGs2iSHsU0iv3rs2A7tqsJRi3pIFgAZVX/ulivbi+ulms8b/WtzfAFZdEkbNO8ALiTcszEF1TLPtOWxS+YdYq2qqpoyBJ+lrJTIc4HpRsVhc36+Xt6utihVeLNSz9X+oCH5/WGkUlrWwkdUFiU5DYlGBi/gdB4hxJJdB/TY4BjMNYzND8n0j1oP/e5NuQdPfAaF4fHs5NHKAcdPlPq38SOfSdst2jkulY+rZyG6qINn5sBJkuLC2xwZ2FpdGNmGdmk1UISszRZPQHOEUmMwtF/yAdoe6XrKY3XC1535YLIbhIqng5ArQNyfYMLl6MxErTIqLO0U8r/DWGRNEilXgZ5bvcBrbicVi0VTpaLYda7RU3XXLJxZr+u6dtQT8HBZQamo13ji2BHwXtFLoyMgx8Xduwum/mqm8Va6i1ML572WhUttFOwTQ9NMQtRboxkrrWEeImQ3S3DaJlHJPFM2sRaLQeKOkzKyhExJid2dckLvqSxDoidlm/Zkxi8kwYVGZNkxmC84GiuOj62IbPlmfG2oonJm90YwtPhXP0Qcapi7DlGgny0F2ZZsCdoKYN0TKB7Ewcp4veUY02Abw2OHfAIdaqQHvqFi0Ky5S/I6hvwsoUT7xYKxdKAWq11g52uX+GYpv3vkxccXtyUzk5+vnqa2KQcaQkvjOQyDvaJgNVimIBVgJyrv2Rx72q5v+AuEOeVSPnnjqNIdN5k1QOOljYi9Z6Jihxz+XakTunsYaCO300/46HkzlxVTa2e9fRzvc6jm38Ax/WeIy/TrSzsLvD8aCbWihy6GFfaEthBqIOqp2GOio0IgIBRg1Rgv1ApBBcwllb19ZqObStbdC2jqCc7cE/COL2VFMbI7h1efnu8B2omwZRH46JzTSd9D9fXVitDxhQ7odA4x3ke05oma/puApe82cjbCsi8bPEtZulILZ+lprwvVq9f0MXZTkZ2Uxvhe73ROH0MfhZu2dRxhRtAmeN4O89wId7kBBiN0j1PAAojJzRFWZMAHcAzywNGSCtBStkwMNhUIV0wH7P+HHkC0wL5uIx7T2v7rCHZfCGmuikwXEwLsR7gXOGx+GcnkyG+1k8iehodgB3L3b2HmGqZm7jj5Yfh4H8yDCNki3VJ5MLqDshhM/UvUAbLxxLo9ETwDsmn54BxYR1kpaHPrR2BcwnI3FSkDsrdcpP62Z8qqDu1MPDWPQ6ua0o6J5YFlzfZSDH9+EcZjm0CybNgQ7NKfG8UCWgbGZGgLj6ODersZHoKMLuoHD1UZvqNu9HnoeADhzpwOWhk41HvAXemRMuI37gtv4bdKTo1+zX2Z4pA7G3xYwQQBiItl6+t5oPK9tkbTgwVx8hfYUe8hIThkTbPkvR3/dq3LAeSbNDAca8UxIMwONK0RydYn2NDir63d1Ue8FSTI/mUZQ7is1dFMOlFNJJn4f+eprdQE+QHYERyPCbRT1wDQQXYtc3gOPO7MAMIwvBOj3v5LG7nNPTHwF8H3C3ebjCZ24mtJoyUpYallGRxPO5aWVzGNJjPYZVBAowP3JReUuEvTPOC/l8RMZ4IX1TxtCF5mYEO5QTThlvyuK9mrtWckTQqXvibBpPJ3kaRRfYBvxrQ3rDWxdEfwkGI1jxmbcV2yU1faZ17neubpa3KTJTjsrjDwmRhb5SzCS6/2BJ9Vkxkw/oQ2KfYEKGQVJKsoMXF0INBFIlRDXMzAeTobmeH7ugatZs78bnLv5l5bmOfqCwrG4LAm55cYpY+Zv5TADch58MPEzzMc1FBgj/9zeIZChbgza3dg6Lg735aPmwORiUD0/jpoeb+we7dTCYAcq3h+qpSakJUBbOpE4k5GQwuBpKveh4ZwGVTjzd9psniE9iX2S+ET1cKczcivmTebVs4+DqBcDo+v0do6uZUsu+6ZKf8RhoaKahWfHoENgbX14hpSH0LXhX5b8duXquyTN0KBp+CJ14V3F3EfS4dZXBQeW/IaE8RzHGuuYwji1uW4DRX0g42newFQAA")))
if "src.classification.train_distilroberta" in sys.modules:
    importlib.reload(sys.modules["src.classification.train_distilroberta"])
if "src.classification.eval_transformer" in sys.modules:
    importlib.reload(sys.modules["src.classification.eval_transformer"])
print("SUCCESS: Model pipeline modules deployed and reloaded.")

# 3. Resolve uploaded dataset files
TRAIN_CSV = TARGET_DATA_DIR / "apple_support_intent_training_candidates.csv"
GOLDEN_CSV = TARGET_DATA_DIR / "apple_support_intent_golden_set.csv"
BASELINES_JSON = TARGET_DATA_DIR / "apple_support_intent_evaluation_results.json"

search_roots = [Path("/content"), Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive")]

# Scan for any uploaded apple_support folder anywhere in /content
print("Scanning for uploaded AppleSupport datasets in /content...")
found_folders = []
for root in search_roots:
    if root.exists():
        for p in root.glob("**/*"):
            if p.is_dir() and "apple" in p.name.lower() and "support" in p.name.lower():
                if p.resolve() != TARGET_DATA_DIR.resolve():
                    found_folders.append(p)

for folder in set(found_folders):
    print(f"Found uploaded dataset folder at: {folder}")
    for item in folder.iterdir():
        if item.is_file():
            dest = TARGET_DATA_DIR / item.name
            if not dest.exists() or dest.stat().st_size != item.stat().st_size:
                shutil.copy(item, dest)
                print(f"  Copied {item.name} ({item.stat().st_size / (1024*1024):.2f} MB) -> {dest}")

# Also search directly by file names across search roots if needed
if not TRAIN_CSV.exists():
    for root in search_roots:
        if root.exists():
            matches = [p for p in root.rglob("*apple_support_intent_training_candidates.csv") if p.resolve() != TRAIN_CSV.resolve()]
            if matches:
                shutil.copy(matches[0], TRAIN_CSV)
                print(f"Copied training candidates from {matches[0]}")
                break

if not GOLDEN_CSV.exists():
    for root in search_roots:
        if root.exists():
            matches = [p for p in root.rglob("*apple_support_intent_golden_set.csv") if p.resolve() != GOLDEN_CSV.resolve()]
            if matches:
                shutil.copy(matches[0], GOLDEN_CSV)
                print(f"Copied golden set from {matches[0]}")
                break

if not BASELINES_JSON.exists():
    for root in search_roots:
        if root.exists():
            matches = [p for p in root.rglob("*apple_support_intent_evaluation_results.json") if p.resolve() != BASELINES_JSON.resolve()]
            if matches:
                shutil.copy(matches[0], BASELINES_JSON)
                print(f"Copied baselines json from {matches[0]}")
                break
    if not BASELINES_JSON.exists():
        BASELINES_JSON.write_bytes(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/+1cW4/jthV+31/BzkPRIp4J75Q2KJobgiyQBEUT9KUNBI5Nj9WVRFUXO06w/72HvowtjSnbM5Z3vWvPxrFNivx0eG7k+aA/XiF0k8e5SeLMRFNTlLHNbl6jG3KHbwausdK/2cym8+1Gum4sdDayaVQaM4LfOV38aKY6qXUFfaMqTk1Z6TRfXIWpvMXhLRG/YPmakNeE3XHFCMefYfwar4ZM7cgkJfT/A7657/q/toireXSvywXKx6Z15yjTqfv15sdVV/RNossSfb2+YLDufgwygaUK2Ray9ghlnsSVu/rBJiMDI8J4m46VrXQSrZoKM7TFyN0UEcFjl2FiSzOKZrZIRtFqYPi+3Vk8drYgfp0kUWqqIh6WW0KARj0c1oUezuFXfIdDzAabtlQPCxvlMGq8Wj7oggP6pAv0gAnWQ4RP2sdk2UYE2Wqbmfhh4mB3zvDYqzkJ29VlM8+q8d2jEHJTRHFWmazyysHW0Aj/N2WzzV3dRDjYbtvG1WgYk9sS1sPsaCrrPLeFUwFxhx8b3m3dk87zyBZgHsU0HpooLsvanA8VxR5Y97qqTDGPcjszxRnxSB+eOAE7fYhyPU9hbc+HCMx8N6KhTfNEg6pFY3Bt93r49oygiBdUlplhFU+dN8xMBY7jjLCUB9XILHR7oovRTBdnVO/AA2hsdFUXgMjOosqecd185gbeHPxWOYnzHLT8fIBCD57Sjiu3VFGdjyDinA8Qox5EdfY2s7MsstVkr0OCqEGpCLDkgkhCw934SBc+IoUMMKQeTFEVEBz4VpT7/Ho73noQLSMn0tN9qw7RkhDCmQwFJlxK5pV62PwjHatAIIUhASAiQmHFFPHdpvDFr3VMPugOILUhMhCKc0yk5B130LGArTvgoeQyUCFRgkPeERx2B0/yBvCc49phjVINacNvzawh0ffLjPPf26O3conGzDvjeqNHM8Q2m1rRrtG4I/C02nfEgEaPtj9uirfpGxttLTfVFHXLZTQam9b72PJrww5WYt8W8fZnhBr2+sF8FVvffh1cGHiKLxq9vGT0hF00enLJ6NUlgw8uWm8u2uOElwye0YtWHL6N/lX70yaVKxOX3SRmanxnYel9/FDbuozSOqni1XlRO3tdHM+5LUkjk8nsYjt0883iZO52cTKHNkd+6KfPv/oCPZ7UodLkuoBPyRxNY41clhmPTDY0n5sS8t3lNTrTybyMy7ub3VuJCu7hvl6cKE7LqJyneWVTH1zKW9lgAZn1IgMetHLXrf0JJ2r3sQvktyMNSeooLpdSWyD2za08U+OOqXdv0GEr4DLbxUnnFLJk34ytfP8FUy5UwWXd9WLgnbOJU01WGNgblaB0IM6p8YoTe+ZT/vnI7vnqxERO+cA0qrNoTjmBfVeU2bicH6stwj+dgg1wxyZuY9hDXZoyWhtWY1fRxDIzporiRWmCYEGaEq/Mb4vj+y+/yvPE/LzcTDojRolGWQ1mjvSwAoWJf9fD+D81xmOWDRB4kweL8sLeJybVJXIuwF0Ft6eL1CwGMAn6OqlNZWFLNHDD5Rq0IYEPpUFVoe81mqPMIuhexNnIuC6wYzSFm8WM9F1zgwXqNIKlNaONM2ttutp7xJUf2nEQsmmEvXDxEGc7+lQ2j1Zzgj9o74ybQl5ccAisndAa7e8Gh87StR19Og9+9jwdW+IDpvHF45tJnWpXFJrGZgbju6jjhHzzxhmJzYZxokG50MJp3S7RIHeQ5Gpaa7VC2ycBA1QOC2MyNIb3380Ags4IrY4g0KjQMbTYJLEzuAtoQ0uxoVlcTWwNY6MSGmDGkU3jDFQVrcLQ7pjls7FQCuWxse/NHDXs7A59r0coBXSL4wrEUGqzalIukGv01ixaTGJzd/qExhpkMUATuCaukAsecKs2RWVlC7AoO9LzxZWgdWBUw4nO4jJF92YSr350A8Yl2NwMlZnOYdQ79JNdz/43lDjh2jGKs6FNzRdwDfSGf4DQZhDfC/O/OgaTgLlscoe+hYgJ9+HWZoT+jGZOZpVFJUh0CBY/BBm/QQ+mQuM6SeDiMUTav6MfNAg1h94lcoauGRtQgf90NfarsTeN/d44s8pQPoEAB5kkWp/moZFO9QMEGV1DgopGdeEsGhIOHRcr09nSrG3tQ5Dk5XVljjNpgqVUgu+06R/nYJZTV3A3LghWzny+JEQEIgAQFJMA7M3MUPyPCUTHtc8Bm3L2uLhgBmpjnImtXVWiy6pEFCNwQ4C1fLRpJ5ciA3vKwCkUyJ16lsh9vNrO1XaatuOCGgJvDVZTOg1xfJC1PQ11BRpW2HwSD1vx8S/bulf+dal5Lpfd6N5Yx0kNCpvbsrpdCnTbml5tb1lXortJ7ENcVvEQbuIBNgVlc3/V4q/88t3tm2+/Q5+hH1ZXoX9urnoeiYVLFaoLJLGIkIk9JBappOgisUhJqY/EIhkJ95NYZLBdxtpJYmnhbJNYRCh5rywWScUzCqBKhpThx3fWM62F+KvIzA+SSyJYsHmXPdJcgoAyQUNOiJJcYemDK1qS86OXoZKKcxJIzKUgtFdOTOjD20RLO6StAipxIEMhBSWK902Y8Ra4qWr++RHTALYbfPPeP52mNSHx3UNnv8Y9dPY8MfdGtl4+9Ep0aAkW26YieqTm8CZa5YXrR+t1GSeh6gRKeDE1X0GHAjT8jgj6pPIs6RYb9oXXyXVIFKuQyc17j0SfplzAO73c2sSLeD4u1POABQqHUilOj6f5uIyFSYaZVAQDVq/XlpQETCkRwC6KhR1e2+UxjArKZMAJ57xHho9LhiCckRCgcR74Q6RXTE+WA5Ij6rpjxoVSV27PR8vtER9kzfHAWjVpXiWPKmse00q767stGKeo9hJxDL6jvrYpGacQ/Z7q/ekWZo8ceii0k87FflFlvQe9OeqrPB34984ukS/Roh50/kUsqfet8yfUanUKye+hI3a28qOoOt2ilme3WNLbwhB65fW8iNfD/HwJ8fERe7rIKELIk3N7OuajfXB7RMdyKti79sHvCTumDK8Enw+V4LN/N9gqqsHWnO6tYLoj4YCcqoi5F6MP5zPLjMcUTfEdoQ0ae6910wZ/9cr9uXJ/Tsb9OeDUaFsTBaPqAC/ARHhCL9CJ0IfymabZdcS20wsoJp851d6jwJ2uQHJ5pQZdBDWo4yD3ib5ycoBVEcbYqazKD84H8EyxjmMcnNWgCA8/Hb5QnBkNEpqmh9CEFp3Rz//68ZnsoIBiJi+RHcQDvo8dJDu5QZRwHzdIhPKAB9xIwfc94KaF8gk3SDzGwH64QaL5ks8hCjUqq5z1/vyboPXyVRl5R1lREEU5YSpkxFUW+6QNscbL+8yDbRqTe+9CTyUjQgUhV66o2ydvqIOWJVqIOzSkb6YQ9ZbJSdD86yg1U4Ibgj0DWcjPDjmCHtQnH6iDvtLFAPqEOT8dBtNJrjpoTU9C8eGMNv7JZ3B8uGRKBpCthxyShl4pPq3Xyw2Gv+xRPtvEFbKDgXwAS0ZiTEJClu9ebYH0g2MZYAEuKQhFJ0cGC1gQzGDMQPTJ73FP2MEhFYQEYHpe7H4ZPQ3EVEJPFoQyCMHtXgk+V4LPBRB8gv6eaNM/waddfsXnIviQ59Xc6ekYCy+padOzsx3o6Srw7H0/surK8LkyfPpm+HTrGP2wGT64m+GjrgyfT5jh00Gz4UReH91zSfSejgkVDWQf9J6gi1F0Jfd8JOQeLuQh5B5G8Psl9zic5yH38GdP1HE4sLP+qGh4JfdcyT0fALmHSn6AE8BhEL5Hbg+V/FzUHinIWZkIgRBXas9FUHs6zzxby8ooZoeYFWXyVGZ1LOfGQXyuUR3FI6KhVOdly1H58ZN7Xrn/3v0fW1zRR2xrAAA=")))
        print("Unpacked Phase 9 baseline evaluation results.")

assert TRAIN_CSV.exists(), f"Training candidates CSV missing at {TRAIN_CSV}"
assert GOLDEN_CSV.exists(), f"Golden evaluation set CSV missing at {GOLDEN_CSV}"

# 4. Load datasets and verify strict mathematical isolation
import pandas as pd
from src.classification.build_golden_evaluation_set import verify_training_isolation

df_train = pd.read_csv(TRAIN_CSV)
df_golden = pd.read_csv(GOLDEN_CSV)

verify_training_isolation(df_train, df_golden)
print(f"SUCCESS: Loaded {len(df_train)} training candidates and {len(df_golden)} golden evaluation records.")
print(f"Training dataset size: {TRAIN_CSV.stat().st_size / (1024*1024):.2f} MB")
print(f"Golden dataset size: {GOLDEN_CSV.stat().st_size / 1024:.2f} KB")
print("Strict training isolation verified: S_train ∩ S_golden = ∅")

In [ ]:
# Cell 4: Fine-Tune DistilRoBERTa on GPU
import importlib
import src.classification.train_distilroberta as train_module
importlib.reload(train_module)

OUTPUT_DIR = "models/distilroberta_intent_classifier"

print("Starting GPU Fine-Tuning Pipeline for DistilRoBERTa on Tesla T4...")
model, tokenizer, metadata = train_module.train_distilroberta(
    train_csv_path=TRAIN_CSV,
    golden_csv_path=GOLDEN_CSV,
    output_dir=OUTPUT_DIR,
    model_name="distilroberta-base",
    epochs=3,
    batch_size=32,
    learning_rate=3e-5,
    max_length=128,
    seed=42,
)

print("")
print("Fine-Tuning Complete!")
print(f"Model artifacts saved to: {OUTPUT_DIR}")

In [ ]:
# Cell 5: Comprehensive Evaluation on Golden Set & Comparison with Baselines
import json
from src.classification.train_distilroberta import evaluate_transformer_on_golden_set, get_label_mappings
from src.classification.build_golden_evaluation_set import load_golden_evaluation_set
from src.classification.eval_transformer import compare_with_phase9_baselines

golden_df = load_golden_evaluation_set(GOLDEN_CSV)
_, id2label = get_label_mappings()

eval_results = evaluate_transformer_on_golden_set(
    model=model,
    tokenizer=tokenizer,
    golden_df=golden_df,
    id2label=id2label,
    max_length=128,
)

# Compare with Phase 9 baselines
BASELINES_JSON = "data/processed/apple_support/apple_support_intent_evaluation_results.json"
comparison = compare_with_phase9_baselines(eval_results, BASELINES_JSON)

# Save comprehensive results JSON
RESULTS_JSON = "data/processed/apple_support/apple_support_distilroberta_evaluation_results.json"
with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump({"evaluation": eval_results, "comparison": comparison}, f, indent=2)

print(f"Evaluation results saved to: {RESULTS_JSON}")

# Print Comparison Table
print("\n=== Comparative Evaluation on Golden Set (155 Closed-World Records) ===")
print(f"{'Model':32s} | {'Accuracy':<10s} | {'Macro-F1':<10s} | {'Weighted-F1':<12s}")
print("-" * 70)
for m_key, m_info in comparison["models"].items():
    m_name = m_info["name"]
    met = m_info["metrics"]
    acc = met.get('accuracy', 0.0)
    mf1 = met.get('macro_f1', 0.0)
    wf1 = met.get('weighted_f1', 0.0)
    print(f"{m_name:32s} | {acc:<10.4f} | {mf1:<10.4f} | {wf1:<12.4f}")

# Print Per-Intent Deltas
print("\n=== Per-Intent F1 Comparison vs. Logistic Regression (Phase 9 Best) ===")
print(f"{'Intent':25s} | {'DistilRoBERTa':<14s} | {'LogReg':<10s} | {'Delta F1':<10s}")
print("-" * 65)
for intent, d in sorted(comparison["per_intent_deltas_vs_logistic_regression"].items()):
    sign = "+" if d['delta_f1'] >= 0 else ""
    print(f"{intent:25s} | {d['distilroberta_f1']:<14.4f} | {d['logistic_regression_f1']:<10.4f} | {sign}{d['delta_f1']:<10.4f}")

# Print Slice Performance
print("\n=== DistilRoBERTa Performance Across 8 Difficulty Slices ===")
print(f"{'Difficulty Slice':26s} | {'Accuracy':<10s} | {'Correct/Total':<15s}")
print("-" * 55)
for s_name, s_info in sorted(eval_results["slice_level_metrics"].items()):
    if "accuracy" in s_info:
        print(f"{s_name:26s} | {s_info['accuracy']:<10.4f} | {s_info['correct']}/{s_info['total']}")
    else:
        print(f"{s_name:26s} | {'N/A':<10s} | -/{s_info['total']} (ambiguous)")

# Print Ambiguous Cases Diagnostic
print("\n=== Ambiguous Cases Diagnostic (3 needs_review records) ===")
for amb in eval_results["ambiguous_cases_analysis"]:
    top_s = ", ".join(f"{t['intent']} ({t['confidence']:.3f})" for t in amb['top_predictions'])
    print(f"Tweet {amb['tweet_id']}:")
    print(f"  Text:            {amb['text'][:85]}...")
    print(f"  Top Prediction:  {amb['predicted_intent']} (confidence: {amb['confidence']:.4f}, margin: {amb['confidence_margin']:.4f})")
    print(f"  Top Predictions: {top_s}")
    print(f"  Notes:           {amb['human_reviewer_notes']}")

In [ ]:
# Cell 6: Package & Export Model Artifacts
import shutil

archive_name = "distilroberta_intent_classifier_artifacts"
print(f"Creating archive: {archive_name}.zip...")
shutil.make_archive(archive_name, 'zip', OUTPUT_DIR)
print(f"SUCCESS: Archive created at {archive_name}.zip")

try:
    from google.colab import files
    print("Downloading artifacts to local machine...")
    files.download(f"{archive_name}.zip")
    files.download(RESULTS_JSON)
except Exception as e:
    print(f"Note: If running in VS Code Colab extension, files are already in workspace: {OUTPUT_DIR}")